In [2]:
import pandas as pd
import calendar

path = "../data/raw/DADS5250_Hackathon_Google_Data.xlsx"

forecast = pd.read_csv("../data/processed/forecast.csv")
params = pd.read_excel(path, sheet_name="Agent Salary and Threshold")
staff = pd.read_excel(path, sheet_name="Existing Agent Staffing").set_index("Country")["Existing_Agent_Count"]

print(forecast.shape, params.shape, staff.shape)

(264, 6) (22, 3) (22,)


In [3]:
import calendar

SUPPORT_DAYS = 60     # an advertiser is supported for 60 days
MAX_WAIT_DAYS = 60    # leaves the pool after waiting 60 days
CAPACITY = 10         # advertisers per agent at once
UPLIFT = 0.135        # expected uplift: 13.5% of annual budget (assumption A1)
FIRE_COST = 0.40      # firing costs 40% of annual salary

salaries = params.set_index("Country")["Annual_Agent_Salary_USD"]

In [11]:
def run_plan(country, headcount, scenario="base"):
    salary = salaries[country]
    fc = forecast[forecast["country"] == country].set_index("month")
    budget = fc["avg_eligible_budget"].iloc[0]
    start_agents = staff[country]

    pool = []
    active = []
    rows = []
    day = 0

    for month in range(1, 13):
        agents = headcount[month - 1]
        arrivals_per_day = fc.loc[month, f"daily_eligible_{scenario}"]
        n_days = calendar.monthrange(2026, month)[1]
        served = abandoned = wait_days = busy = revenue = 0.0          # NEW: revenue starts at 0

        for _ in range(n_days):
            active = [g for g in active if day - g[0] < SUPPORT_DAYS]

            still_waiting = []
            for g in pool:
                if day - g[0] >= MAX_WAIT_DAYS:
                    abandoned += g[1]
                else:
                    still_waiting.append(g)
            pool = still_waiting

            pool.append([day, arrivals_per_day])

            free = agents * CAPACITY - sum(g[1] for g in active)
            while free > 1e-9 and pool:
                g = pool[0]
                take = min(free, g[1])
                waited = day - g[0]
                active.append([day, take])
                served += take
                wait_days += take * waited
                revenue += take * budget * UPLIFT * (1 - waited / MAX_WAIT_DAYS)   # NEW: less uplift the longer they waited
                g[1] -= take
                free -= take
                if g[1] <= 1e-9:
                    pool.pop(0)

            busy += sum(g[1] for g in active)
            day += 1

        prev = start_agents if month == 1 else headcount[month - 2]
        hires = max(agents - prev, 0)
        fires = max(prev - agents, 0)
        cost = (agents * salary / 12
                + hires * salary / 12
                + fires * salary / 12
                + fires * FIRE_COST * salary)

        rows.append({
            "month": month, "agents": agents, "hires": hires, "fires": fires,
            "served": served, "abandoned": abandoned,
            "avg_wait_days": wait_days / served if served else 0,
            "utilization": busy / (agents * CAPACITY * n_days) if agents else 0,
            "waiting_end": sum(g[1] for g in pool),
            "revenue": revenue,                                          # NEW: use the running total
            "cost": cost,
        })

    return pd.DataFrame(rows)

In [27]:
plans = {
    "Keep all 652":       [652] * 12,
    "Peak need 296":      [296] * 12,
    "Too few: 200":       [200] * 12,
}

results = []
for name, plan in plans.items():
    m = run_plan("USA", plan)
    results.append({
        "plan": name,
        "revenue_M": round(m["revenue"].sum() / 1e6, 1),
        "cost_M": round(m["cost"].sum() / 1e6, 1),
        "net_M": round((m["revenue"].sum() - m["cost"].sum()) / 1e6, 1),
        "served": round(m["served"].sum()),
        "abandoned": round(m["abandoned"].sum()),
        "max_wait_days": round(m["avg_wait_days"].max(), 1),
    })
pd.DataFrame(results)

,plan,revenue_M,cost_M,net_M,served,abandoned,max_wait_days
0,Keep all 652,151.3,50.7,100.7,13043,0,0.0
1,Peak need 296,151.3,36.4,115.0,13043,0,0.0
2,Too few: 200,127.0,32.5,94.5,11705,0,24.3



Keep all 652 - $100.7M Everyone served, but you pay for ~350 idle agents all year 

Peak need 296  - $115.0M Everyone served, and $14M cheaper. Best of the three. 

Too few: 200 - they bring in less revenue because of waiting up to 24 days and fewer get served 

In [10]:
  run_plan("USA", [296] * 12).round(1)

,month,agents,hires,fires,served,abandoned,avg_wait_days,utilization,waiting_end,revenue,cost
0,1,296,0,356,1024.0,0.0,0.0,0.2,0,11881580.2,15290311.4
1,2,296,0,0,980.1,0.0,0.0,0.5,0,11371843.8,1917118.0
2,3,296,0,0,1023.5,0.0,0.0,0.7,0,11875778.6,1917118.0
3,4,296,0,0,972.0,0.0,0.0,0.7,0,11278218.7,1917118.0
4,5,296,0,0,943.0,0.0,0.0,0.6,0,10941728.6,1917118.0
5,6,296,0,0,858.5,0.0,0.0,0.6,0,9961266.2,1917118.0
6,7,296,0,0,914.5,0.0,0.0,0.6,0,10611040.1,1917118.0
7,8,296,0,0,919.0,0.0,0.0,0.6,0,10663254.1,1917118.0
8,9,296,0,0,1063.0,0.0,0.0,0.6,0,12334101.3,1917118.0
9,10,296,0,0,1498.5,0.0,0.0,0.8,0,17387253.8,1917118.0


 

Three things to notice in the monthly table- 

1. January’s cost is $18.3M, compared with $1.3M in other months. That’s the firing penalty: 452 agents × 40% of salary, plus their notice month. You pay it once, and it pays for itself because those agents would otherwise cost about $35M in salary over the year. 

2. January utilization is only 0.3 (30%). That’s because of the rule that agents start 2026 with no advertisers, so caseloads take a few weeks to fill. It’s a side effect of the starting assumption, not a real problem. 

3. “Too few” looks slightly worse than it really is. 1,337 advertisers are still waiting at the end of December. They’d be served in January 2027, but the simulation stops at Dec 31, so their revenue isn’t counted. Its a known limitation

In [7]:
run_plan("Vietnam", [59] * 12).round(1)

,month,agents,hires,fires,served,abandoned,avg_wait_days,utilization,waiting_end,revenue,cost
0,1,59,0,0,382.5,0.0,0.0,0.3,0.0,893459.4,88028.0
1,2,59,0,0,207.5,0.0,0.0,0.9,164.4,484687.1,88028.0
2,3,59,0,0,370.2,0.0,14.2,1.0,181.3,864638.1,88028.0
3,4,59,0,0,219.8,0.0,14.1,1.0,338.9,513508.4,88028.0
4,5,59,0,0,382.5,0.0,27.2,1.0,319.4,893459.4,88028.0
5,6,59,0,0,219.8,0.0,26.9,1.0,439.6,513508.4,88028.0
6,7,59,0,0,383.4,0.0,37.4,1.0,408.2,895667.1,88028.0
7,8,59,0,0,231.2,0.0,36.6,1.0,482.4,540121.9,88028.0
8,9,59,0,0,372.1,0.0,44.6,1.0,501.9,869053.6,88028.0
9,10,59,0,0,230.3,0.0,41.9,1.0,830.6,537914.2,88028.0


In [25]:
def net_value(country, plan, scenario="base"):
    m = run_plan(country, plan, scenario)
    return m["revenue"].sum() - m["cost"].sum()

def make_plan(base, q4):
    return [base] * 9 + [q4] * 3

<b>Define the Optimizer </b>

In [29]:
def optimize_country(country, scenario="base"):
    fc = forecast[forecast["country"] == country]
    peak = int((fc["daily_eligible_base"] * 6).max())
    high = int(peak * 1.2) + 1
    step = max(1, peak // 30)

    # Round 1: coarse search
    best = None
    for base in range(0, high, step):
        for q4 in range(base, high, step):
            v = net_value(country, make_plan(base, q4), scenario)
            if best is None or v > best[0]:
                best = (v, base, q4)

    # Round 2: fine search around the best
    _, b0, q0 = best
    for base in range(max(0, b0 - step), b0 + step + 1):
        for q4 in range(max(base, q0 - step), q0 + step + 1):
            v = net_value(country, make_plan(base, q4), scenario)
            if v > best[0]:
                best = (v, base, q4)

    return make_plan(best[1], best[2])

In [30]:
plan = optimize_country("USA")
print(plan)
run_plan("USA", plan).round(1)

[204, 204, 204, 204, 204, 204, 204, 204, 204, 289, 289, 289]


,month,agents,hires,fires,served,abandoned,avg_wait_days,utilization,waiting_end,revenue,cost
0,1,204,0,448,1024.0,0.0,0.0,0.3,0.0,11881580.2,18150444.2
1,2,204,0,0,980.1,0.0,0.0,0.8,0.0,11371843.8,1321257.0
2,3,204,0,0,1023.5,0.0,0.0,1.0,0.0,11875778.6,1321257.0
3,4,204,0,0,972.0,0.0,0.0,1.0,0.0,11278218.7,1321257.0
4,5,204,0,0,943.0,0.0,0.0,0.9,0.0,10941728.6,1321257.0
5,6,204,0,0,858.5,0.0,0.0,0.9,0.0,9961266.2,1321257.0
6,7,204,0,0,914.5,0.0,0.0,0.9,0.0,10611040.1,1321257.0
7,8,204,0,0,919.0,0.0,0.0,0.9,0.0,10663254.1,1321257.0
8,9,204,0,0,1063.0,0.0,0.0,0.9,0.0,12334101.3,1321257.0
9,10,289,85,0,1498.5,0.0,0.0,0.8,0.0,17387253.8,2422304.5


In [31]:
rows = []
for country in sorted(forecast["country"].unique()):
    plan = optimize_country(country)
    best = run_plan(country, plan)
    keep = run_plan(country, [staff[country]] * 12)
    rows.append({
        "country": country,
        "existing": staff[country],
        "base_jan_sep": plan[0],
        "q4_oct_dec": plan[9],
        "net_M": round((best["revenue"].sum() - best["cost"].sum()) / 1e6, 2),
        "net_keep_all_M": round((keep["revenue"].sum() - keep["cost"].sum()) / 1e6, 2),
        "max_wait_days": round(best["avg_wait_days"].max(), 1),
        "abandoned": round(best["abandoned"].sum()),
    })
    print("done:", country)

summary = pd.DataFrame(rows)
summary["gain_M"] = summary["net_M"] - summary["net_keep_all_M"]
print(summary[["existing", "base_jan_sep", "q4_oct_dec", "net_M", "net_keep_all_M", "gain_M"]].sum())
summary

done: Argentina
done: Australia
done: Brazil
done: Canada
done: China
done: France
done: Germany
done: India
done: Indonesia
done: Italy
done: Japan
done: Mexico
done: Poland
done: South Africa
done: South Korea
done: Spain
done: Thailand
done: Turkey
done: UAE
done: UK
done: USA
done: Vietnam
existing          6474.00
base_jan_sep      1646.00
q4_oct_dec        2320.00
net_M              475.23
net_keep_all_M     363.99
gain_M             111.24
dtype: float64


,country,existing,base_jan_sep,q4_oct_dec,net_M,net_keep_all_M,max_wait_days,abandoned,gain_M
0,Argentina,200,14,20,-0.63,-2.38,0.4,0,1.75
1,Australia,450,71,106,12.67,4.67,0.2,0,8.00
2,Brazil,189,91,125,44.61,43.51,0.0,0,1.10
3,Canada,228,72,106,19.94,15.72,0.0,0,4.22
4,China,252,91,130,52.22,50.67,0.0,0,1.55
5,France,489,66,89,9.62,-4.16,0.2,0,13.78
6,Germany,602,83,119,24.18,11.04,0.1,0,13.14
7,India,428,230,320,85.00,83.99,0.0,0,1.01
8,Indonesia,120,29,40,4.36,3.68,0.1,0,0.68
9,Italy,350,79,114,11.22,5.70,0.3,0,5.52


In [32]:
plan_rows = []
for country in summary["country"]:
    row = summary[summary["country"] == country].iloc[0]
    for month in range(1, 13):
        agents = row["base_jan_sep"] if month <= 9 else row["q4_oct_dec"]
        plan_rows.append({"country": country, "month": month, "agents": agents})

pd.DataFrame(plan_rows).to_csv("../data/processed/plan.csv", index=False)
summary.to_csv("../data/processed/plan_summary.csv", index=False)
print("Saved!")

Saved!


In [33]:
countries = params.merge(staff.reset_index(), on="Country")
countries.to_csv("../data/processed/countries.csv", index=False)
countries.head()

,Country,Annual_Agent_Salary_USD,Advertiser_Eligibility_Threshold_USD,Existing_Agent_Count
0,USA,77721,35000,652
1,UK,51650,27000,589
2,Germany,51328,26000,602
3,India,14118,13000,428
4,Brazil,27950,29000,189
